[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bmcguir2/astromol/blob/refactor/docs/notebooks/03_custom_views.ipynb)

# Custom filtered views

Use `CensusView.filtered(...)` when the normal census boundary is correct, but you want to analyze a special subset of the database.

In [ ]:
import subprocess
import sys

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "git+https://github.com/bmcguir2/astromol.git@refactor",
    ])

In [ ]:
from pathlib import Path

from IPython.display import Image, display

from astromol.census import CensusView
from astromol.database import Database
from astromol.figures import (
    cumulative_detection_data,
    du_histogram_data,
    periodic_heatmap_data,
    write_cumulative_detections_plot,
    write_du_bar_chart,
    write_periodic_heatmap,
)

db = Database()
view = CensusView.for_census(db, "2026")
out = Path("astromol_custom_view_outputs")
out.mkdir(exist_ok=True)

## Carbon-bearing molecules

A molecule filter receives a molecule record and returns `True` if the molecule should remain in the view.

In [ ]:
carbon_view = view.filtered(
    molecule_filter=lambda molecule: molecule.atom_counts.get("C", 0) > 0
)

print("All ISM/CSM molecules:", len(view.ism_molecules()))
print("Carbon-bearing ISM/CSM molecules:", len(carbon_view.ism_molecules()))

path = out / "carbon_periodic_heatmap.png"
write_periodic_heatmap(periodic_heatmap_data(carbon_view), path)
display(Image(filename=str(path)))

## Millimeter detections

A detection filter receives a detection record and returns `True` if the detection should remain in the view.

In [ ]:
mm_view = view.filtered(
    detection_filter=lambda detection: "mm" in detection.wavelengths
)

print("All ISM/CSM detections:", len(view.ism_detections()))
print("Millimeter ISM/CSM detections:", len(mm_view.ism_detections()))

path = out / "mm_cumulative_detections.png"
write_cumulative_detections_plot(cumulative_detection_data(mm_view), path)
display(Image(filename=str(path)))

## Source-type subset

Filters can inspect linked sources, telescopes, references, or molecule metadata.

In [ ]:
dark_cloud_view = view.filtered(
    detection_filter=lambda detection: any(
        source.type == "Dark Cloud"
        for source in detection.sources
    )
)

path = out / "dark_cloud_du_bar_chart.png"
write_du_bar_chart(du_histogram_data(dark_cloud_view), path)
display(Image(filename=str(path)))